In [1]:
import pandas as pd
from pathlib import Path
from itertools import product, combinations
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

In [2]:
def read_data(TASKS = None, EXPERIMENTS = None, EMBS = None, MYTH_COMBOS = None):
    """Load all CSVs into a dict keyed by (task, experiment, emb, myth_combo)."""
    
    if TASKS is None: TASKS = ["2_AdviceGeneration", "3_Summarization"]
    if EXPERIMENTS is None: EXPERIMENTS = ["2_SemanticShift_Cosine", "3_MythShift_Projection"]
    if EMBS is None: EMBS = ["W2V", "GLOVE", "SBERT"]
    if MYTH_COMBOS is None: MYTH_COMBOS = ["singles", "pairs"]
        
    datasets = {}
    
    for task, experiment, emb, myth_combo in product(TASKS, EXPERIMENTS, EMBS, MYTH_COMBOS):
        data = pd.read_csv(f'{task}/SampleResults/{experiment}/{emb}_{myth_combo}.csv')
        datasets[(task, experiment, emb, myth_combo)] = data
    return datasets

In [3]:
MODELS = ["gemma", "llama", "mistral", "phi", "qwen"]
PROMPTS = ["p1", "p2", "p3"]

MYTH_TYPES = ["clothing", "victim_intoxication", "perpetrator_intoxication", "resistance"]
MYTH_PAIRS = [f"{a}+{b}" for a, b in combinations(sorted(MYTH_TYPES), 2)]
MYTH_COL = {"singles": "myth_type", "pairs": "myth_pair"}

FRAMES = ["NegMyth", "NegNonMyth", "PosMyth", "PosNonMyth"]
DOSES  = [1, 2]

In [4]:
datasets = read_data()

In [5]:
"""Type-II ANOVA on cohens_dz per (task, exp), with emb and combo as factors."""

'Type-II ANOVA on cohens_dz per (task, exp), with emb and combo as factors.'

In [10]:
def run_anova(datasets):

    results = {}
    for task in ["2_AdviceGeneration", "3_Summarization"]:
        for exp in ["2_SemanticShift_Cosine", "3_MythShift_Projection"]:
            frames = []
            for (t, e, emb, combo), df in datasets.items():
                if t != task or e != exp: continue
                frames.append(df.assign(emb=emb, combo=combo))
            combined = pd.concat(frames, ignore_index=True).dropna(subset=["cohens_dz"])

            factors = ["C(model)", "C(myth_type)", "C(frame)", "C(dose)", "C(emb)", "C(combo)"]
            if "prompt_variant" in combined.columns and task == "2_AdviceGeneration":
                factors.append("C(prompt_variant)")

            formula = "cohens_dz ~ " + " + ".join(factors)
            fit    = ols(formula, data=combined).fit()
            table  = anova_lm(fit, typ=2)[["sum_sq", "df", "F", "PR(>F)"]]
            table["eta_sq"]     = table["sum_sq"] / table["sum_sq"].sum()
            table["sig"]        = table["PR(>F)"].apply(
                lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else " "))
                if pd.notna(p) else "")
            
            # model fit summary as separate row block
            fit_summary = pd.DataFrame([{
                "sum_sq": None, "df": fit.df_model, "F": fit.fvalue,
                "PR(>F)": fit.f_pvalue, "eta_sq": None,
                "sig": "***" if fit.f_pvalue < 0.001 else "",
                "R2": fit.rsquared, "R2_adj": fit.rsquared_adj
            }], index=["MODEL_FIT"])
            table["R2"]     = None
            table["R2_adj"] = None

            out = pd.concat([table, fit_summary])
            results[(task, exp)] = out
            out.index = out.index.str.replace("_", " ")
            out.columns = out.columns.str.replace("_", " ")
            out = out.round(4)
            out.to_csv(f"Tables/ANOVA/{task}_{exp}.txt")
    return results

In [11]:
results = run_anova(datasets)

/tmp/ipykernel_16034/2324808960.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat([table, fit_summary])
/tmp/ipykernel_16034/2324808960.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat([table, fit_summary])
/tmp/ipykernel_16034/2324808960.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To reta